# Phase 6 — Soil Module Joint Multi-Task Training

**Backbone:** EfficientNet-B0 at 224×224, mixed precision

**Heads:** `soil_type` (7) | `moisture_appearance` (3) | `texture` (3)

**Datasets:** Phantom-fs (1,188) | Sirajganj 2025 (1,177) | IRSID+VIT merged (279)

**Expected wall-time on Colab T4:** ~6–12 hours total (1–2 sessions)

Per master plan §22. Resume-aware via HF Hub checkpoints to
`ankit-iiitdmj/iks-soil-multitask` (private). Each batch mixes samples
from all 3 source datasets via `ConcatDataset` + shuffle; per-sample
loss masking with `ignore_index=-1` makes each sample supervise
exactly one of the three heads.

## ⚠️ Before you start
- Set the runtime to **GPU** (Runtime → Change runtime type → T4 GPU).
- **Colab free tier:** ~12 h continuous session, ~3–4 h daily GPU
  quota. Training is checkpoint-resumable so a disconnect mid-run is
  safe — re-run from Cell 9 and it picks up the latest epoch.
- HF Hub Write token required (Cell 3).


In [ ]:
# Cell 2 — setup: clone repo + install dependencies (defensive pattern)
REPO_URL = "https://github.com/ankit8453/iks-rag-thesis.git"
REPO_PATH = "/content/iks-rag-thesis"

import os
import subprocess
import sys

# Defensive: if a prior partial clone exists without requirements.txt, wipe it.
if os.path.isdir(REPO_PATH) and not os.path.isfile(os.path.join(REPO_PATH, "requirements.txt")):
    print(f"Removing partial clone at {REPO_PATH} ...")
    subprocess.run(["rm", "-rf", REPO_PATH], check=True)
if not os.path.isdir(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)

os.chdir(REPO_PATH)
print("Repo root contents:", sorted(os.listdir(".")))

# Install runtime deps. Colab pre-installs torch / numpy / pandas / sklearn,
# so we skip those and avoid version conflicts.
_pip_packages = [
    "timm>=1.0.0",
    "albumentations>=1.4",
    "huggingface_hub>=0.24",
    "datasets>=2.20",
    "iterative-stratification>=0.1.7",
    "pydantic>=2.7",
    "pyyaml>=6.0",
]
proc = subprocess.run(
    [sys.executable, "-m", "pip", "install", *_pip_packages],
    capture_output=True, text=True,
)
if proc.returncode != 0:
    print("PIP STDOUT (tail):\n" + proc.stdout[-3000:])
    print("PIP STDERR (tail):\n" + proc.stderr[-3000:])
    raise SystemExit("pip install failed — see tails above.")
print("setup ok")


In [ ]:
# Cell 3 — HF Hub login
from huggingface_hub import login, HfApi
login()  # Colab shows an inline widget — paste your Write token

_whoami = HfApi().whoami()
assert _whoami["name"] == "ankit-iiitdmj", (
    f"HF Hub token belongs to {_whoami['name']!r}, expected 'ankit-iiitdmj'."
)
print(f"HF Hub ok: user={_whoami['name']}")


In [ ]:
# Cell 4 — GPU check + auto batch size
import subprocess, torch, sys
sys.path.insert(0, REPO_PATH)

subprocess.run(["nvidia-smi"], check=False)
print()
if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(0)
    print(f"GPU: {dev.name}, VRAM: {dev.total_memory / 1024**3:.1f} GiB")
else:
    print("No GPU detected — switch runtime to GPU before continuing.")

from src.soil.train import auto_batch_size
batch = auto_batch_size()


## Training configuration

- 30 epochs total (5 warmup with frozen backbone, then 25 full unfrozen)
- AdamW optimizer: `lr=1e-4`, `weight_decay=1e-4`
- Cosine annealing schedule (`T_max=30`)
- Joint loss: `1.0*L_soil_type + 1.0*L_moisture + 1.0*L_texture` with `ignore_index=-1`
- Per-epoch checkpoint pushed to `ankit-iiitdmj/iks-soil-multitask` (private)
- Resume-aware: if a checkpoint exists on HF Hub, training continues from there


In [ ]:
# Cell 6 — Load 3 HF datasets (train + val + test for each)
from datasets import load_dataset

DATASETS = {
    "soil_type": "ankit-iiitdmj/iks-soil-phantomfs",
    "moisture":  "ankit-iiitdmj/iks-soil-sirajganj-moisture",
    "texture":   "ankit-iiitdmj/iks-soil-texture-irsid-vit",
}

loaded = {head: load_dataset(repo) for head, repo in DATASETS.items()}
for head, dsd in loaded.items():
    repo = DATASETS[head].split("/")[-1]
    print(f"{repo}: train={len(dsd['train'])} val={len(dsd['val'])} test={len(dsd['test'])}")


In [ ]:
# Cell 7 — Transforms + DataLoaders (single combined train loader, 3 per-task val/test)
import numpy as np
import torch
import yaml
from torch.utils.data import ConcatDataset, DataLoader, Dataset

from src.soil.dataset import build_multitask_labels
from src.soil.transforms import build_soil_eval_aug, build_soil_train_aug

with open("configs/data/soil_norm.yaml") as fh:
    norm = yaml.safe_load(fh)
MEAN = tuple(norm["mean"])
STD  = tuple(norm["std"])
IMG_SIZE = 224

train_aug = build_soil_train_aug(IMG_SIZE, MEAN, STD)
eval_aug  = build_soil_eval_aug(IMG_SIZE, MEAN, STD)


class _MultiTaskWrapper(Dataset):
    """Wrap an HF dataset split as multi-task rows (one head valid, others -1)."""

    def __init__(self, hf_split, head: str, transform):
        self.hf_split = hf_split
        self.head = head
        self.transform = transform

    def __len__(self):
        return len(self.hf_split)

    def __getitem__(self, idx):
        row = self.hf_split[idx]
        arr = np.asarray(row["image"].convert("RGB"))
        tensor = self.transform(image=arr)["image"]
        labels = build_multitask_labels(
            self.head, int(row["label_idx"]),
            head_order=("soil_type", "moisture", "texture"),
        )
        return {
            "image": tensor,
            "soil_type_label": torch.tensor(labels["soil_type"], dtype=torch.long),
            "moisture_label":  torch.tensor(labels["moisture"],  dtype=torch.long),
            "texture_label":   torch.tensor(labels["texture"],   dtype=torch.long),
        }


train_datasets = [
    _MultiTaskWrapper(loaded["soil_type"]["train"], "soil_type", train_aug),
    _MultiTaskWrapper(loaded["moisture"]["train"],  "moisture",  train_aug),
    _MultiTaskWrapper(loaded["texture"]["train"],   "texture",   train_aug),
]
train_loader = DataLoader(
    ConcatDataset(train_datasets),
    batch_size=batch, shuffle=True, num_workers=2, pin_memory=True,
)

val_loaders = {
    head: DataLoader(
        _MultiTaskWrapper(loaded[head]["val"], head, eval_aug),
        batch_size=batch, shuffle=False, num_workers=2, pin_memory=True,
    )
    for head in ("soil_type", "moisture", "texture")
}
test_loaders = {
    head: DataLoader(
        _MultiTaskWrapper(loaded[head]["test"], head, eval_aug),
        batch_size=batch, shuffle=False, num_workers=2, pin_memory=True,
    )
    for head in ("soil_type", "moisture", "texture")
}

print(f"train loader: {len(train_loader.dataset)} samples in {len(train_loader)} batches")
for head in ("soil_type", "moisture", "texture"):
    print(f"  val {head}:  {len(val_loaders[head].dataset)} samples")
    print(f"  test {head}: {len(test_loaders[head].dataset)} samples")


In [ ]:
# Cell 8 — Build SoilMultiTaskClassifier
from src.soil.model import SoilMultiTaskClassifier

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SoilMultiTaskClassifier(
    backbone_name="efficientnet_b0", pretrained=True, dropout=0.3,
).to(device)

bb = model.backbone_param_count()
hd = model.head_param_count()
print(f"backbone params: {bb:,}")
print(f"head params:     {hd:,}")
print(f"total params:    {bb + hd:,}")


In [ ]:
# Cell 9 — optimizer + scheduler + scaler + checkpoint manager
import torch
from src.soil.train import SoilCheckpointManager

EPOCHS_TOTAL = 30
FREEZE_EPOCHS = 5

optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-4, weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_TOTAL)
scaler = torch.amp.GradScaler("cuda")
ckpt_mgr = SoilCheckpointManager(repo_id="ankit-iiitdmj/iks-soil-multitask")

start_epoch = 0
history: list[dict] = []
best_val_sum = 0.0

_resume = ckpt_mgr.try_load_latest()
if _resume is not None:
    print(f"Resuming from epoch {_resume['epoch']} ...")
    model.load_state_dict(_resume["model_state"])
    if _resume.get("optimizer_state") is not None:
        optimizer.load_state_dict(_resume["optimizer_state"])
    if _resume.get("scheduler_state") is not None:
        scheduler.load_state_dict(_resume["scheduler_state"])
    if _resume.get("scaler_state") is not None:
        scaler.load_state_dict(_resume["scaler_state"])
    start_epoch = int(_resume["epoch"]) + 1
    history = list(_resume.get("history", []))
    if history:
        best_val_sum = max(h.get("val_sum", 0.0) for h in history)
    print(f"  history len={len(history)}  best_val_sum={best_val_sum:.4f}")
else:
    print("No prior checkpoint — starting fresh from ImageNet weights.")


In [ ]:
# Cell 10 — training loop
import time, datetime
from src.soil.train import train_one_epoch, evaluate_per_task

for epoch in range(start_epoch, EPOCHS_TOTAL):
    is_frozen = epoch < FREEZE_EPOCHS
    if is_frozen:
        model.freeze_backbone()
        stage_tag = "FROZEN"
    else:
        model.unfreeze_backbone()
        stage_tag = "UNFROZEN"

    t0 = time.time()
    train_losses = train_one_epoch(model, train_loader, optimizer, scaler, device)
    val_metrics  = evaluate_per_task(model, val_loaders, device)
    scheduler.step()
    elapsed = time.time() - t0

    val_sum = (
        val_metrics["soil_type"]["top1_accuracy"]
        + val_metrics["moisture"]["top1_accuracy"]
        + val_metrics["texture"]["top1_accuracy"]
    )

    print(
        f"[epoch {epoch+1}/{EPOCHS_TOTAL}] {stage_tag} | "
        f"type_loss={train_losses['loss_soil_type']:.4f} "
        f"moist_loss={train_losses['loss_moisture']:.4f} "
        f"tex_loss={train_losses['loss_texture']:.4f} "
        f"total={train_losses['loss_total']:.4f} | "
        f"val: type_acc={val_metrics['soil_type']['top1_accuracy']:.4f} "
        f"moist_acc={val_metrics['moisture']['top1_accuracy']:.4f} "
        f"tex_acc={val_metrics['texture']['top1_accuracy']:.4f} | "
        f"{elapsed:.0f}s"
    )

    history.append({
        "epoch": int(epoch + 1),
        "stage": stage_tag,
        **train_losses,
        "val_soil_type": val_metrics["soil_type"],
        "val_moisture":  val_metrics["moisture"],
        "val_texture":   val_metrics["texture"],
        "val_sum": float(val_sum),
        "elapsed_seconds": float(elapsed),
        "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    })

    ckpt_mgr.save(
        epoch=epoch,
        model_state=model.state_dict(),
        optimizer_state=optimizer.state_dict(),
        scheduler_state=scheduler.state_dict(),
        scaler_state=scaler.state_dict(),
        history=history,
        val_metrics=val_metrics,
    )

    if val_sum > best_val_sum:
        best_val_sum = float(val_sum)
        ckpt_mgr.save_best(
            epoch=epoch, model_state=model.state_dict(),
            val_metrics=val_metrics,
        )
        print(f"  -> new best (val_sum={best_val_sum:.4f}) checkpoint_best.pt pushed")

best_epoch = max(history, key=lambda h: h["val_sum"]) if history else None
if best_epoch:
    print(
        f"\nTraining complete. best epoch={best_epoch['epoch']} "
        f"val_sum={best_epoch['val_sum']:.4f} "
        f"type={best_epoch['val_soil_type']['top1_accuracy']:.4f} "
        f"moist={best_epoch['val_moisture']['top1_accuracy']:.4f} "
        f"tex={best_epoch['val_texture']['top1_accuracy']:.4f}"
    )


## Test-set evaluation

Mirrors Phase 5 fix: evaluates on **test** splits (held out, never seen
during training). Pushes results as `eval_metrics_test.json` to keep
separate from any val-time metrics file.


In [ ]:
# Cell 12 — held-out test-set evaluation per task
import json, torch
from huggingface_hub import HfApi
from sklearn.metrics import f1_score

# Pull the best checkpoint and load it before test-set eval.
_best = ckpt_mgr.try_load_latest()  # latest is most recent epoch
# If you've also pushed checkpoint_best.pt, prefer that one:
try:
    from huggingface_hub import hf_hub_download
    _best_path = hf_hub_download(
        repo_id=ckpt_mgr.repo_id, filename="checkpoint_best.pt", repo_type="model",
    )
    _best = torch.load(_best_path, map_location=device, weights_only=False)
    print(f"Loaded checkpoint_best.pt (epoch {_best['epoch']})")
except Exception as exc:
    print(f"Falling back to checkpoint_latest.pt — {exc}")
model.load_state_dict(_best["model_state"])
model.eval()

summary = {}
for head in ("soil_type", "moisture", "texture"):
    loader = test_loaders[head]
    y_true, y_pred, top5_correct = [], [], 0
    n_classes = {"soil_type": 7, "moisture": 3, "texture": 3}[head]
    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device, non_blocking=True)
            labels = batch[f"{head}_label"].to(device, non_blocking=True)
            logits = model(images)[head]
            preds = logits.argmax(dim=1)
            y_true.extend(int(v) for v in labels.tolist())
            y_pred.extend(int(v) for v in preds.tolist())
            k = min(5, n_classes)
            top5 = logits.topk(k, dim=1).indices
            top5_correct += int((top5 == labels.unsqueeze(1)).any(dim=1).sum().item())
    n = len(y_true)
    top1 = sum(1 for t, p in zip(y_true, y_pred) if t == p) / max(1, n)
    top5 = top5_correct / max(1, n) if n_classes >= 5 else 1.0
    macro = float(f1_score(y_true, y_pred, average="macro", zero_division=0.0))
    nice_name = {"soil_type": "soil_type", "moisture": "moisture_appearance", "texture": "texture"}[head]
    print(f"\n=== {nice_name} TEST eval ===")
    print(f"  top1 acc: {top1:.4f}")
    if n_classes >= 5:
        print(f"  top5 acc: {top5:.4f}")
    else:
        print(f"  top5 acc: 1.0000   ({n_classes} classes → top-5 trivially perfect)")
    print(f"  macro F1: {macro:.4f}")
    print(f"  n_samples: {n}")
    summary[head] = {
        "top1_accuracy": top1,
        "top5_accuracy": top5,
        "macro_f1": macro,
        "n_samples": n,
        "num_classes": n_classes,
    }

tmp_path = "/tmp/eval_metrics_test.json"
with open(tmp_path, "w") as fh:
    json.dump(summary, fh, indent=2)
HfApi().upload_file(
    path_or_fileobj=tmp_path,
    path_in_repo="eval_metrics_test.json",
    repo_id=ckpt_mgr.repo_id,
    repo_type="model",
)
print("\nTest evaluation complete. Metrics pushed to HF Hub.")
